<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/10_full_scaled_quantization_aware_unlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1: Environment Setup & Data Pipelines

In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
import math
import copy
import gc

# 1. Define the Dataset Class
class MUSE_Dataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encodings = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze()
        }

# 2. Load Tokenizer
model_path = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Load Data from Drive
forget_df = pd.read_csv("/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set.csv")
retain_df = pd.read_csv("/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/retain_set.csv")

forget_texts = forget_df['text'].tolist()
retain_texts = retain_df['text'].tolist()

# 4. Initialize DataLoaders (Batch Size = 2 for safety on 16GB/24GB GPUs)
BATCH_SIZE = 2

forget_dataset = MUSE_Dataset(forget_texts, tokenizer)
forget_dataloader = DataLoader(forget_dataset, batch_size=BATCH_SIZE, shuffle=True)

retain_dataset = MUSE_Dataset(retain_texts, tokenizer)
retain_dataloader = DataLoader(retain_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"✅ Loaded {len(forget_texts)} forget records and {len(retain_texts)} retain records.")
print("✅ DataLoaders initialized.")

Loading Tokenizer...
✅ Loaded 889 forget records and 3555 retain records.
✅ DataLoaders initialized.


## Cell 2: Load Model & Apply Scaled Surgical Freezing

In [3]:
print("Loading Phi-3 Model in 16-bit to GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    local_files_only=True,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

# Freeze entire model
for param in model.parameters():
    param.requires_grad = False

# Target a block of layers for scaled unlearning
target_modules = []
for i in range(28, 32):
    mlp_module = model.model.layers[i].mlp
    mlp_module.to(torch.float32) # Upcast targeted MLPs to FP32 for gradient stability
    for param in mlp_module.parameters():
        param.requires_grad = True
    target_modules.append(mlp_module)

print("✅ Model frozen. Layers 28-31 MLPs are unfrozen.")

# Helper function to extract flat weights across multiple modules
def get_flat_weights(modules):
    return torch.cat([p.view(-1) for m in modules for p in m.parameters()])

# Store original weights for the grid penalty calculation
original_block_weights = get_flat_weights(target_modules).clone().detach()

# Initialize Optimizer
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4) # Slightly lower LR for batching
print("✅ Optimizer Initialized.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Phi-3 Model in 16-bit to GPU...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model frozen. Layers 28-31 MLPs are unfrozen.
✅ Optimizer Initialized.


## Cell 3: Vectorized Multi-Objective Loss Function

In [4]:
class ScaledUnlearningLoss(nn.Module):
    def __init__(self, grid_margin=0.133, lambda_reg=50.0, alpha_retain=50.0):
        super().__init__()
        self.grid_margin = grid_margin
        self.lambda_reg = lambda_reg
        self.alpha_retain = alpha_retain

    def forward(self, forget_logits, forget_labels, retain_logits, retain_labels, current_weights, original_weights):
        # 1. FORGET LOSS (Gradient Ascent)
        shift_f_logits = forget_logits[..., :-1, :].contiguous().float()
        shift_f_labels = forget_labels[..., 1:].contiguous()
        f_ce_loss = F.cross_entropy(shift_f_logits.view(-1, shift_f_logits.size(-1)), shift_f_labels.view(-1))
        forget_loss = -torch.clamp(f_ce_loss, max=50.0)

        # 2. RETAIN LOSS (Standard Language Modeling)
        shift_r_logits = retain_logits[..., :-1, :].contiguous().float()
        shift_r_labels = retain_labels[..., 1:].contiguous()
        retain_loss = F.cross_entropy(shift_r_logits.view(-1, shift_r_logits.size(-1)), shift_r_labels.view(-1), ignore_index=tokenizer.pad_token_id)

        # 3. GRID PENALTY (Quantization-Aware Margin)
        weight_diff = torch.abs(current_weights - original_weights)
        grid_penalty = torch.relu(self.grid_margin - weight_diff).mean()

        # 4. TOTAL COMBINED LOSS
        total_loss = forget_loss + (self.alpha_retain * retain_loss) + (self.lambda_reg * grid_penalty)

        return total_loss, forget_loss, retain_loss, grid_penalty

criterion = ScaledUnlearningLoss()
print("✅ Vectorized Loss Function Ready.")

✅ Vectorized Loss Function Ready.


## Cell 4: The Dual-Batch Training Engine

In [5]:
scaler = torch.amp.GradScaler('cuda')
epochs = 5 # Reduced epochs because we are now training on a full dataset
print("--- Starting Scaled Utility-Preserving Unlearning ---")

model.train()
best_grid_penalty = float('inf')
best_weights = None

for epoch in range(epochs):
    retain_iter = iter(retain_dataloader)
    epoch_loss = 0.0

    for step, forget_batch in enumerate(forget_dataloader):
        optimizer.zero_grad()

        # Get Forget Batch
        forget_inputs = forget_batch['input_ids'].to("cuda")
        forget_attention = forget_batch['attention_mask'].to("cuda")
        forget_labels = forget_inputs.clone()

        # Get Retain Batch
        try:
            retain_batch = next(retain_iter)
        except StopIteration:
            retain_iter = iter(retain_dataloader)
            retain_batch = next(retain_iter)

        retain_inputs = retain_batch['input_ids'].to("cuda")
        retain_attention = retain_batch['attention_mask'].to("cuda")
        retain_labels = retain_inputs.clone()

        with torch.autocast("cuda", dtype=torch.float16):
            # Forward pass 1
            forget_outputs = model(input_ids=forget_inputs, attention_mask=forget_attention)
            # Forward pass 2
            retain_outputs = model(input_ids=retain_inputs, attention_mask=retain_attention)

            current_block_weights = get_flat_weights(target_modules)

            # Calculate Vectorized Loss
            loss, f_loss, r_loss, g_penalty = criterion(
                forget_outputs.logits,
                forget_labels,
                retain_outputs.logits,
                retain_labels,
                current_block_weights,
                original_block_weights
            )

        if math.isnan(loss.item()):
            print(f"⚠️ WARNING: NaN detected at Epoch {epoch+1}, Step {step}! Stopping.")
            break

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        # Track best weights based on grid penalty compliance
        if g_penalty.item() < best_grid_penalty:
            best_grid_penalty = g_penalty.item()
            best_weights = [copy.deepcopy(m.state_dict()) for m in target_modules]

    print(f"Epoch {epoch+1:02d}/{epochs} | Avg Loss: {epoch_loss/len(forget_dataloader):.2f} | Last Grid Penalty: {g_penalty.item():.4f}")

# Restore best weights
if best_weights is not None:
    for idx, m in enumerate(target_modules):
        m.load_state_dict(best_weights[idx])
    print(f"\n✅ Restored model to safely checkpointed weights. Final Penalty: {best_grid_penalty:.4f}")

--- Starting Scaled Utility-Preserving Unlearning ---
Epoch 01/5 | Avg Loss: 103.45 | Last Grid Penalty: 0.1313
Epoch 02/5 | Avg Loss: 90.83 | Last Grid Penalty: 0.1306
Epoch 03/5 | Avg Loss: 77.82 | Last Grid Penalty: 0.1300
Epoch 04/5 | Avg Loss: 67.71 | Last Grid Penalty: 0.1294
Epoch 05/5 | Avg Loss: 58.17 | Last Grid Penalty: 0.1290

✅ Restored model to safely checkpointed weights. Final Penalty: 0.1290


## Cell 5: Automated Dataset-Level Evaluation

In [6]:
def evaluate_scaled_performance(model, forget_loader, retain_loader):
    model.eval()
    print("--- 📊 SCALED EVALUATION ---\n")

    # 1. Evaluate Forget Perplexity
    forget_loss_total = 0.0
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for batch in forget_loader:
            f_inputs = batch['input_ids'].to("cuda")
            f_attn = batch['attention_mask'].to("cuda")
            loss = model(input_ids=f_inputs, attention_mask=f_attn, labels=f_inputs).loss
            forget_loss_total += loss.item()

    avg_forget_loss = forget_loss_total / len(forget_loader)

    # Handle massive perplexity bounds safely to avoid math overflow
    try:
        forget_ppl = math.exp(avg_forget_loss)
    except OverflowError:
        forget_ppl = float('inf')

    # 2. Evaluate Retain Perplexity (Calculate over a sample subset to save time)
    retain_loss_total = 0.0
    eval_batches = min(50, len(retain_loader))
    retain_iter = iter(retain_loader)

    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for _ in range(eval_batches):
            batch = next(retain_iter)
            r_inputs = batch['input_ids'].to("cuda")
            r_attn = batch['attention_mask'].to("cuda")
            loss = model(input_ids=r_inputs, attention_mask=r_attn, labels=r_inputs).loss
            retain_loss_total += loss.item()

    avg_retain_loss = retain_loss_total / eval_batches
    retain_ppl = math.exp(avg_retain_loss)

    print(f"🗑️  Average Forget Set Perplexity: {forget_ppl}")
    print(f"🛡️  Average Retain Set Perplexity: {retain_ppl:.2f}")
    print("\n-------------------------------------------")

evaluate_scaled_performance(model, forget_dataloader, retain_dataloader)

--- 📊 SCALED EVALUATION ---

🗑️  Average Forget Set Perplexity: 745.5338480569874
🛡️  Average Retain Set Perplexity: 4905.54

-------------------------------------------
